In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
# load Fashion-MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# scale to [0, 1] and add channel dimension
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)  # (N, 28, 28, 1)
x_test  = np.expand_dims(x_test, -1)
num_classes = 10


In [ ]:
def make_cnn_fmnist2():
    inputs = keras.Input(shape=(28, 28, 1))

    x = inputs

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model



In [ ]:
model = make_cnn_fmnist2()

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    verbose=2,
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
print("Final train acc:", history.history["accuracy"][-1])
print("Final val acc:", history.history["val_accuracy"][-1])


In [ ]:
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("x_train min/max:", x_train.min(), x_train.max())
print("x_test min/max:", x_test.min(), x_test.max())

print("Label set:", np.unique(y_train))


In [ ]:
def compute_brightness_stats(x, alpha=0.05):
    """
    x: array of shape (N, H, W, C) in [0, 1]
    """
    # mean over pixels and channels per image
    b = x.mean(axis=(1, 2, 3))
    m = b.mean()
    q_low, q_high = np.quantile(b, [alpha, 1 - alpha])
    return b, m, q_low, q_high


In [ ]:
def t_of_tau(tau, m, q_low, q_high):
    if tau < 0:
        return m + tau * (m - q_low)      # move toward dark (low quantile)
    elif tau > 0:
        return m + tau * (q_high - m)     # move toward bright (high quantile)
    else:
        return m


In [ ]:
def apply_brightness_shift(x, delta):
    # x in [0,1], add scalar delta, clip to [0,1]
    x_shifted = x + delta
    return np.clip(x_shifted, 0.0, 1.0)


In [ ]:
def brightness_stress_experiment_all_classes(model, x_test, taus, alpha=0.05):
    """
    Runs the brightness stress experiment for all classes at once and
    returns the portion of predicted class c vs τ, for each class.

    model: trained Keras model with softmax outputs
    x_test: (N, H, W, C) in [0,1]
    taus: iterable of τ values (e.g., np.linspace(-1,1,21))
    alpha: quantile parameter (like in your paper)

    Returns:
        portions: np.array of shape (len(taus), num_classes)
                  portions[i, c] = portion of predictions == c at τ_i
    """
    # Brightness statistics (same as in your single-class function)
    b, m, q_low, q_high = compute_brightness_stats(x_test, alpha=alpha)

    num_classes = model.output_shape[-1]

    portions = np.zeros((len(taus), num_classes))

    for t_idx, tau in enumerate(taus):
        t_tau = t_of_tau(tau, m, q_low, q_high)
        delta = t_tau - m

        x_proj = apply_brightness_shift(x_test, delta)

        # data_augmentation is off here because model is in inference mode
        probs = model.predict(x_proj, verbose=0)
        preds = probs.argmax(axis=1)

        # portion of predicted class c for each class
        for c in range(num_classes):
            portions[t_idx, c] = (preds == c).mean()

    return portions


In [ ]:
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
classes_idx = [2,4,5,9]

In [ ]:
CLASSES = ["Pullover", "Coat", "Sandal", "Ankle boot"]

In [ ]:
taus = np.linspace(-1, 1, 21)

portions = brightness_stress_experiment_all_classes(
    model,
    x_test,
    taus,
    alpha=0.05
)

In [ ]:
portions1 = portions[:,classes_idx]

In [ ]:
def plot_brightness_stress_all_classes(taus, portions, class_names):
    """
    Plot portion of predicted class c vs τ for each class
    in a 2 x 2 grid (4 classes).

    taus: (T,) array of τ values
    portions: (T, num_classes) array
    class_names: list of length num_classes
    """

    num_classes = portions.shape[1]
    assert num_classes == 4, "This plotting layout assumes 4 classes."

    fig, axes = plt.subplots(2, 2, figsize=(6, 6), constrained_layout=True, dpi=300)
    axes = axes.flatten()

    zero_idx = np.where(taus == 0)[0][0]

    for c in range(num_classes):
        ax = axes[c]

        ax.plot(taus, portions[:, c])
        ax.plot(taus[zero_idx], portions[zero_idx, c],
                marker="*", color="red")

        ax.set_title(class_names[c], fontsize=10)

        # y-label only on first column
        if c % 2 == 0:
            ax.set_ylabel("Portion of 1s")

        # x-label only on bottom row
        if c // 2 == 1:
            ax.set_xlabel(r"$\tau$")

        # x-ticks only on bottom row
        if c // 2 != 1:
            ax.set_xticks([])


In [ ]:
plot_brightness_stress_all_classes(taus, portions1, CLASSES)
plt.savefig('brightness_stress.png', bbox_inches="tight")

In [ ]:
def plot_brightness_stress_all_classes(taus, portions, class_names):
    """
    Plot portion of predicted class c vs τ for each class
    in a 2 x 3 grid (6 classes).

    taus: (T,) array of τ values
    portions: (T, num_classes) array
    class_names: list of length num_classes
    """
    num_classes = portions.shape[1]
    assert num_classes == 6, "This plotting layout assumes 6 classes."

    fig, axes = plt.subplots(3, 2, figsize=(5.5, 8), constrained_layout=True, dpi=300)
    axes = axes.flatten()

    zero_idx = np.where(taus == 0)[0][0]

    for c in range(num_classes):
        ax = axes[c]

        ax.plot(taus, portions[:, c])
        ax.plot(taus[zero_idx], portions[zero_idx, c],
                marker="*", color="red")

        ax.set_title(class_names[c], fontsize=10)

        # y-label only on first column
        if c % 2 == 0:
            ax.set_ylabel("Portion of 1s")

        # x-label only on bottom row
        if c // 2 == 2:
            ax.set_xlabel(r"$\tau$")

        # x-ticks only on bottom row
        if c // 2 != 2:
            ax.set_xticks([])

    plt.tight_layout()

In [ ]:
plot_brightness_stress_all_classes(taus, portions1, CLASSES)
plt.savefig('brightness_stress.png', bbox_inches="tight")

In [ ]:
def plot_brightness_stress_all_classes(taus, portions, class_names=FMNIST_CLASSES):
    """a
    Plot portion of predicted class c vs τ for each class
    in a 2 x 5 grid (10 classes).

    taus: (T,) array of τ values
    portions: (T, num_classes) array from brightness_stress_experiment_all_classes
    class_names: list of length num_classes
    """
    num_classes = portions.shape[1]
    assert num_classes == 10, "This plotting layout assumes 10 classes."

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()

    zero_idx = np.where(taus == 0)[0][0]  # index of original dataset

    for c in range(num_classes):
        ax = axes[c]
        ax.plot(taus, portions[:, c])
        ax.plot(taus[zero_idx], portions[zero_idx, c], marker="*", color="red")
        ax.set_title(class_names[c], fontsize=10)

        # y-label only on first column
        if c % 5 == 0:
            ax.set_ylabel("Portion of 1s")

        # x-label only on bottom row
        if c // 5 == 1:
            ax.set_xlabel(r"$\tau$")

        # x-sticks only on bottom row
        if c // 5 != 1:
            ax.set_xticks([])



    #plt.suptitle("Portion of Predictions per Class vs Brightness Stress", fontsize=14)
    plt.tight_layout()
    #plt.show()


In [ ]:
plot_brightness_stress_all_classes(taus, portions, class_names=FMNIST_CLASSES)
plt.savefig('brightness_stress.png')


In [ ]:
def brightness_stress_experiment(
    model, x_test, taus, alpha=0.05, class_idx=0
):
    """
    model: trained Keras model
    x_test: (N, H, W, C) in [0,1]
    y_test: labels (only needed if you want accuracy curves too)
    taus: iterable of τ values (e.g., np.linspace(-1,1,21))
    alpha: quantile parameter (like in your paper)
    class_idx: which class to track (0..9)
    """
    # Baseline predictions (τ = 0) to define stability
    #probs_base = model.predict(x_test, verbose=0)
    #preds_base = probs_base.argmax(axis=1)

    # Brightness statistics
    b, m, q_low, q_high = compute_brightness_stats(x_test, alpha=alpha)

    mean_prob_curves = []   # mean p(class_idx) vs τ
    stability_curves = []   # fraction preds unchanged vs τ
    acc_curves = []         # optional accuracy vs τ (labels assumed invariant)

    for tau in taus:
        t_tau = t_of_tau(tau, m, q_low, q_high)
        delta = t_tau - m

        x_proj = apply_brightness_shift(x_test, delta)

        # data_augmentation is off here because model is in inference mode
        probs = model.predict(x_proj, verbose=0)
        preds = probs.argmax(axis=1)

        # mean prob for chosen class
        mean_prob_curves.append(probs[:, class_idx].mean())

        # prediction stability vs baseline
        # stability = (preds == preds_base).mean()
        # stability_curves.append(stability)

        # optional: accuracy (if you consider labels preserved)
        # acc = (preds == y_test).mean()
        # acc_curves.append(acc)

    return (
        np.array(mean_prob_curves),
        #np.array(stability_curves),
     #   np.array(acc_curves),
    )


In [ ]:
taus = np.linspace(-1, 1, 21)
mean_prob = brightness_stress_experiment(
    model, x_test, taus, alpha=0.05, class_idx=5  # sandal
)


In [ ]:
mean_prob_dress = brightness_stress_experiment(
    model, x_test, taus, alpha=0.05, class_idx=3  # dress
)

In [ ]:
mean_prob_tro = brightness_stress_experiment(
    model, x_test, taus, alpha=0.05, class_idx=1  # trouser
)

In [ ]:
mean_prob_bag = brightness_stress_experiment(
    model, x_test, taus, alpha=0.05, class_idx=8  # bag
)

In [ ]:
# Baseline predictions (τ = 0) to define stability
probs_base = model.predict(x_test, verbose=0)
preds_base = probs_base.argmax(axis=1)

# Brightness statistics
b, m, q_low, q_high = compute_brightness_stats(x_test, alpha=0.05)

stability_curves = []   # fraction preds unchanged vs τ

for tau in taus:
    t_tau = t_of_tau(tau, m, q_low, q_high)
    delta = t_tau - m

    x_proj = apply_brightness_shift(x_test, delta)
    probs = model.predict(x_proj, verbose=0)
    preds = probs.argmax(axis=1)

    # prediction stability vs baseline
    stability = (preds == preds_base).mean()

    stability_curves.append(stability)

In [ ]:
fig, ax1 = plt.subplots(figsize=(4.8, 3.2), dpi=300)
idx = np.where(taus == 0)[0][0]

ax1.plot(taus, stability_curves)
ax1.plot(taus[idx], stability_curves[idx], marker="*", color="red")
ax1.set_xlabel(r'$\tau$ ')
ax1.set_ylabel('Prediction stability')
plt.tight_layout()
plt.savefig('pred_stab.png')
plt.show()

In [ ]:
fig, (ax1,ax2,ax3,ax4, ax5) = plt.subplots(1,5, figsize=(20,5))

ax1.plot(taus, mean_prob[0], marker='o')
ax1.set_xlabel(r'$\tau$ ')
ax1.set_ylabel('Portion of predict 1s')
#ax1.set_title('Mean predicted probability vs brightness')


ax2.plot(taus, mean_prob_dress[0], marker='o')
ax2.set_xlabel(r'$\tau$ ')
#ax2.set_ylabel('Portion of predict 1s for class = Dress)')
#ax2.set_title('Fraction of unchanged predictions')

ax3.plot(taus, mean_prob_tro[0], marker='o')
ax3.set_xlabel(r'$\tau$ ')
#ax3.set_ylabel('Portion of predict 1s for class = Trouser')
#ax3.set_title('Mean predicted probability vs brightness')

ax4.plot(taus, mean_prob_bag[0], marker='o')
ax4.set_xlabel(r'$\tau$ ')
#ax4.set_ylabel('Portion of predict 1s for class = Bag')
#ax4.set_title('Mean predicted probability vs brightness')

ax5.plot(taus, stability_curves, marker='o')
ax5.set_xlabel(r'$\tau$ ')
ax5.set_ylabel('Prediction stability')
plt.tight_layout()
plt.savefig('bright_stress.png')
plt.show()


In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(taus, mean_prob_dress, marker='o')
plt.xlabel(r'$\tau$ ')
plt.ylabel('Mean p(class=3 DRESS)')
plt.title('Mean predicted probability vs brightness')

plt.subplot(1,2,2)
plt.plot(taus, stability_dress, marker='o')
plt.xlabel(r'$\tau$ (brightness stress)')
plt.ylabel('Prediction stability')
plt.title('Fraction of unchanged predictions')

plt.tight_layout()
plt.show()


In [ ]:
def show_fmnist_examples(x_test, taus, alpha=0.05, n=5):
    """
    x_test: (N,H,W,1) in [0,1]
    taus: list of τ values to visualize (e.g., [-1,0,1])
    """
    # brightness statistics
    b = x_test.mean(axis=(1,2,3))
    m = b.mean()
    q_low, q_high = np.quantile(b, [alpha, 1-alpha])

    def t_of_tau(tau):
        if tau < 0:
            return m + tau*(m - q_low)
        elif tau > 0:
            return m + tau*(q_high - m)
        return m

    # pick n random samples once
    idx = np.random.choice(len(x_test), size=n, replace=False)
    originals = x_test[idx]  # shape (n,H,W,1)

    rows = len(taus)
    fig, axes = plt.subplots(rows, n, figsize=(2*n, 2*rows))

    for r, tau in enumerate(taus):
        t_tau = t_of_tau(tau)
        delta = t_tau - m

        x_proj = np.clip(originals + delta, 0, 1)

        for c in range(n):
            ax = axes[r, c] if rows > 1 else axes[c]
            ax.imshow(x_proj[c].squeeze(), cmap="gray")
            ax.set_xticks([]); ax.set_yticks([])

        axes[r, 0].set_ylabel(f"τ={tau}", rotation=0,
                              labelpad=25, fontsize=12)

    #plt.suptitle("Brightness Stress — Visual Check", fontsize=16)
    plt.tight_layout()
    #plt.show()


In [ ]:
show_fmnist_examples(x_test, taus=[-1,0,1], alpha=0.05, n=10)
#plt.savefig('stressed_imags.png')

In [ ]:
show_fmnist_examples(x_test, taus=[-1,0,1], alpha=0.05, n=10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

def show_fmnist_stress_all_classes(x_test, y_test, alpha=0.05, n=3, taus=[-1, 0, 1]):
    """
    Visualize n random samples per class for τ ∈ {-1,0,1},
    using brightness stress projection.

    x_test: (N,28,28,1) normalized in [0,1]
    y_test: (N,) with class indices
    n: number of samples per class (default 3)
    taus: stress levels (should be exactly [-1,0,1])
    """

    # brightness statistics (like your projection code)
    b = x_test.mean(axis=(1,2,3))
    m = b.mean()
    q_low, q_high = np.quantile(b, [alpha, 1-alpha])

    def t_of_tau(tau):
        if tau < 0:
            return m + tau * (m - q_low)
        elif tau > 0:
            return m + tau * (q_high - m)
        return m

    num_classes = len(FMNIST_CLASSES)
    rows = len(taus)
    cols = num_classes * n  # 10 * 3 = 30
    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))

    for class_idx, class_name in enumerate(FMNIST_CLASSES):
        idxs = np.where(y_test == class_idx)[0]
        chosen = np.random.choice(idxs, size=n, replace=False)
        originals = x_test[chosen]  # shape (n,28,28,1)

        for r, tau in enumerate(taus):
            t_tau = t_of_tau(tau)
            delta = t_tau - m

            x_proj = np.clip(originals + delta, 0, 1)

            for j in range(n):
                col = class_idx*n + j  # map class+sample to global column index
                ax = axes[r, col] if rows > 1 else axes[col]

                ax.imshow(x_proj[j].squeeze(), cmap="gray")
                ax.set_xticks([]); ax.set_yticks([])

                # Top row: write class names
                if r == 0:
                    ax.set_title(class_name, fontsize=10, rotation=45)

                # First column group of each row: label τ
                if col == 0:
                    ax.set_ylabel(f"τ={tau}", fontsize=12)

    plt.tight_layout()
    #plt.show()


In [ ]:
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
#    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
def show_fmnist_stress_all_classes(x_test, y_test, alpha=0.05, n=3, taus=[-1, 0, 1], classes_list = FMNIST_CLASSES):
    # Brightness statistics
    b = x_test.mean(axis=(1,2,3))
    m = b.mean()
    q_low, q_high = np.quantile(b, [alpha, 1-alpha])

    def t_of_tau(tau):
        if tau < 0: return m + tau*(m - q_low)
        elif tau > 0: return m + tau*(q_high - m)
        return m

    num_classes = len(FMNIST_CLASSES)
    rows = len(taus)
    cols = num_classes * n

    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))

    for class_idx, class_name in enumerate(FMNIST_CLASSES):
        idxs = np.where(y_test == class_idx)[0]
        chosen = np.random.choice(idxs, size=n, replace=False)
        originals = x_test[chosen]

        for r, tau in enumerate(taus):
            t_tau = t_of_tau(tau)
            delta = t_tau - m
            x_proj = np.clip(originals + delta, 0, 1)

            for j in range(n):
                col = class_idx*n + j
                ax = axes[r, col] if rows > 1 else axes[col]
                ax.imshow(x_proj[j].squeeze(), cmap="gray")

                # Minimal labels
                ax.set_xticks([]); ax.set_yticks([])

                # if r == 0:
                #     ax.set_title(FMNIST_CLASSES[class_idx], fontsize=8, pad=2)
                # if col == 0:
                #     ax.set_ylabel(f"τ={tau}", fontsize=10, labelpad=2, rotation=0, va="center")

    # 🔥 Remove all whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    fig.tight_layout(pad=0)

    plt.show()


In [ ]:
show_fmnist_stress_all_classes(x_test, y_test, alpha=0.05, n=1, classes_list = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat"])
#plt.savefig('stressed_all_classes.png')

In [ ]:
show_fmnist_stress_all_classes(x_test, y_test, alpha=0.05, n=1, classes_list = ["Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"])
#plt.savefig('stressed_all_classes.png')

In [ ]:
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

def show_fmnist_stress_classes(x_test, y_test, class_list,
                               alpha=0.05, n=3, taus=[-1, 0, 1]):
    """
    Visualize n random samples per *selected* Fashion-MNIST class
    under brightness stress τ ∈ taus (default [-1, 0, 1]).

    x_test: (N, 28, 28, 1) in [0,1]
    y_test: (N,) with class indices
    class_list: list of class names, e.g. ["Sandal", "Sneaker", "Ankle boot"]
    alpha: quantile parameter for brightness projection
    n: number of samples per class
    taus: list of τ values to visualize (rows)
    """

    # Brightness statistics over the whole test set
    b = x_test.mean(axis=(1, 2, 3))
    m = b.mean()
    q_low, q_high = np.quantile(b, [alpha, 1 - alpha])

    def t_of_tau(tau):
        if tau < 0:
            return m + tau * (m - q_low)
        elif tau > 0:
            return m + tau * (q_high - m)
        return m

    # Map requested class names to indices
    class_indices = [FMNIST_CLASSES.index(name) for name in class_list]
    num_classes = len(class_indices)

    rows = len(taus)
    cols = num_classes * n

    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))

    for k, (cls_idx, class_name) in enumerate(zip(class_indices, class_list)):
        # indices of this class in the test set
        idxs = np.where(y_test == cls_idx)[0]
        chosen = np.random.choice(idxs, size=n, replace=False)
        originals = x_test[chosen]  # (n, 28, 28, 1)

        for r, tau in enumerate(taus):
            t_tau = t_of_tau(tau)
            delta = t_tau - m
            x_proj = np.clip(originals + delta, 0, 1)

            for j in range(n):
                col = k * n + j  # which column this sample goes to
                ax = axes[r, col] if rows > 1 else axes[col]

                ax.imshow(x_proj[j].squeeze(), cmap="gray")
                ax.set_xticks([])
                ax.set_yticks([])

                # # Top row gets class titles
                # if r == 0:
                #     ax.set_title(class_name, fontsize=8, pad=1)

    # Tight spacing between images
    plt.subplots_adjust(wspace=0.01, hspace=0.01)
    fig.tight_layout(pad=0.1)

    # # External τ labels on the left
    # for r, tau in enumerate(taus):
    #     fig.text(
    #         0.005,
    #         (rows - r - 0.5) / rows,   # vertically center per row
    #         f"τ = {tau}",
    #         va="center", ha="left", fontsize=12
    #     )

    #plt.show()


In [ ]:
show_fmnist_stress_classes(
    x_test,
    y_test,
    class_list=["Pullover", "Coat"],
    n=1,
    taus=[-1, 0, 1],
    alpha=0.05
)
plt.savefig('stressed_classes1.png')

In [ ]:
show_fmnist_stress_classes(
    x_test,
    y_test,
    class_list=["Sandal", "Ankle boot"],
    n=1,
    taus=[-1, 0, 1],
    alpha=0.05
)
plt.savefig('stressed_classes3.png')

In [ ]:
files.download('stressed_classes3.png')

In [ ]:
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
#    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
# Fashion-MNIST class names
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

In [ ]:
def plot_random_class_samples(x, y, class_names, n=5):
    """
    Plot random sample images for specific Fashion-MNIST classes.

    Args:
        x (np.array): image data (N,28,28,1) in [0,1]
        y (np.array): labels (N,)
        class_names (list): e.g. ["Sandal", "Sneaker", "Ankle boot"]
        n (int): number of samples per class
    """
    FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
    # Convert class names to label indices
    class_indices = [FMNIST_CLASSES.index(c) for c in class_names]

    rows = len(class_indices)
    fig, axes = plt.subplots(rows, n, figsize=(2*n, 2*rows))

    for i, cls_idx in enumerate(class_indices):
        # filter indices for this class
        idxs = np.where(y == cls_idx)[0]
        chosen = np.random.choice(idxs, size=n, replace=False)

        for j in range(n):
            ax = axes[i, j] if rows > 1 else axes[j]
            ax.imshow(x[chosen[j]].squeeze(), cmap="gray")
            ax.set_xticks([])
            ax.set_yticks([])

        axes[i, 0].set_ylabel(
            FMNIST_CLASSES[cls_idx],
            rotation=0, labelpad=50, fontsize=12
        )

    plt.suptitle("Random Samples from Selected Classes", fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Shirt","Sandal",  "Sneaker", "Ankle boot", "Bag"
]

def plot_one_per_class_fmnist(x, y):
    """
    Plots 1 random image from each Fashion-MNIST class
    in a 2 x 5 grid.
    x: (N, 28, 28, 1), y: (N,)
    """
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))

    for class_idx, class_name in enumerate(FMNIST_CLASSES):
        row = class_idx // 5
        col = class_idx % 5

        idxs = np.where(y == class_idx)[0]
        img = x[np.random.choice(idxs)]

        ax = axes[row, col]
        ax.imshow(img.squeeze(), cmap="gray")
        #ax.set_title(class_name)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes.flatten():
        ax.axis("off")

    plt.tight_layout(pad=0.5)


In [ ]:
plot_one_per_class_fmnist(x_test, y_test)
plt.savefig('one_per_class.png')

In [ ]:
files.download('one_per_class.png')

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Get predicted labels
y_prob = model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_test, y_pred,
                            target_names=[
                                "T-shirt/top", "Trouser", "Pullover", "Dress",
                                "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
                            ]))


In [ ]:
model.summary()


In [ ]:
total_params = np.sum([np.prod(w.shape) for w in model.trainable_weights])
print("Total trainable parameters:", total_params)


In [ ]:
print("Final train acc:", history.history["accuracy"][-1])
print("Final val acc:", history.history["val_accuracy"][-1])
